# 06-shuffle-and-broadcast-joins

Compare shuffle and explicit broadcast join plans. Both joins should count 1,000,000 rows; count adds an aggregation to execution.

Run in Databricks with a provided spark session. Run cells from top to bottom. See ../docs/DEMO.md for setup and reset instructions.


In [ ]:

from pyspark.sql import functions as F

# Disable automatic broadcast and AQE
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "8")

# Large table: 1 million transactions
transactions = spark.range(1000000).withColumn(
    "customer_id", F.col("id") % 100
)

# Small table: 100 customers
customers = spark.range(100).withColumnRenamed(
    "id", "customer_id"
)

In [ ]:

# Normal join without broadcast
normal_join = transactions.join(
    customers, "customer_id"
)

# Inspect physical plan
normal_join.explain("formatted")

# Execute the full join and return only its row count
print("Shuffle join rows:", normal_join.count())


In [ ]:

from pyspark.sql.functions import broadcast

# Broadcast small table
broadcast_join = transactions.join(
    broadcast(customers),
    "customer_id"
)

# Check physical plan
broadcast_join.explain("formatted")

# Execute the full join and return only its row count
print("Broadcast join rows:", broadcast_join.count())
